# NISAR and ESA BIOMASS: OVERLAP

Date: February 2, 2026

Authors: Harshini Girish (UAH), Rajat Shinde (UAH), Alex Mandel (Development Seed), Samantha Niemoeller (JPL)

Description: This notebook queries NISAR L2 GCOV granules (via `earthaccess`) and ESA BIOMASS satellite items (via the ESA MAAP STAC API, e.g., `BiomassLevel1b`) for a chosen AOI and time settings. It converts returned items to footprint polygons and plots them on a single interactive Folium map as two toggleable layers. An optional overlap layer highlights where NISAR and BIOMASS footprints intersect (bbox-only or true geometry). The result quickly shows where data coincides spatially to support fusion workflows.


## Run This Notebook

To access and run this tutorial within MAAP's Algorithm Development Environment (ADE), please refer to the ["Getting started with the MAAP"](https://docs.maap-project.org/en/latest/getting_started/getting_started.html) section of our documentation.

Disclaimer: it is highly recommended to run a tutorial within MAAP's ADE, which already includes packages specific to MAAP, such as maap-py. Running the tutorial outside of the MAAP ADE may lead to errors. Additionally, it is recommended to use the `Pangeo` workspace within the ADE, since certain packages relevant to this tutorial are already installed.

## Additional Resources
- [NISAR](https://nisar.jpl.nasa.gov/)
- [BIOMASS](https://docs.maap-project.org/en/develop/science/ESA_CCI/ESA_CCI_V5_Token_Access.html)


## Import and Install Packages

In [8]:
import os
import stat
import getpass
import pathlib

import numpy as np
import matplotlib.pyplot as plt

import earthaccess
from pystac_client import Client

from shapely.geometry import Polygon, box, mapping, shape
from collections import Counter

from folium import Map, GeoJson, LayerControl

plt.rcParams["figure.figsize"] = (6, 6)
plt.rcParams["axes.grid"] = False


## Inputs
This “Inputs” section defines the search settings used later in the notebook.

- **BBOX** sets the area of interest as *(min_lon, min_lat, max_lon, max_lat)* and can be used to spatially filter both datasets.
- **NISAR_TEMPORAL** restricts the NISAR search to granules acquired within that date range, and **NISAR_COUNT** limits how many NISAR granules (and footprints) will be plotted.
- **BIOMASS_COLLECTION** selects the ESA **BIOMASS satellite** STAC collection (e.g., `BiomassLevel1b`), and **BIOMASS_TEMPORAL / BIOMASS_DT** defines the STAC datetime range used in the BIOMASS query.


In [9]:
NISAR_TEMPORAL = ("2025-10-01", "2025-12-31")

NISAR_COUNT = 6

BIOMASS_COLLECTION = "BiomassLevel1b"

BIOMASS_TEMPORAL = ("2025-12-01T00:00:00Z", "2025-12-31T23:59:59Z")
BIOMASS_DT = f"{BIOMASS_TEMPORAL[0]}/{BIOMASS_TEMPORAL[1]}"

print("NISAR_TEMPORAL:", NISAR_TEMPORAL)
print("BIOMASS_COLLECTION:", BIOMASS_COLLECTION)
print("BIOMASS_DT:", BIOMASS_DT)



NISAR_TEMPORAL: ('2025-10-01', '2025-12-31')
BIOMASS_COLLECTION: BiomassLevel1b
BIOMASS_DT: 2025-12-01T00:00:00Z/2025-12-31T23:59:59Z


## Access the Data


### 1) NISAR data

This cell sets the NISAR collection short name (`NISAR_L2_GCOV_BETA_V1`), logs in to Earthdata via `earthaccess.login()`, and then queries CMR for matching NISAR granules within the specified `NISAR_TEMPORAL` window, limited to `NISAR_COUNT` results and filtered to cloud-hosted items. Finally, it prints how many granules were returned by the search.


In [10]:
NISAR_SHORT_NAME = "NISAR_L2_GCOV_BETA_V1"

earthaccess.login()

nisar_results = earthaccess.search_data(
    short_name=NISAR_SHORT_NAME,
    cloud_hosted=True,
    temporal=NISAR_TEMPORAL,
    count=NISAR_COUNT,
)

print("NISAR granules found:", len(nisar_results))


NISAR granules found: 6


This cell defines helper functions used before visualization. `_get_umm(g)` safely extracts the UMM metadata dictionary from an `earthaccess` granule. `nisar_granule_to_feature(g)` then converts a single NISAR granule into a GeoJSON Feature by reading its spatial geometry (preferring a polygon from `GPolygons` and falling back to a `BoundingRectangles` box if needed), extracting the granule’s start/end times, and attaching an ID/title in the feature properties. The output Feature objects are later collected into a FeatureCollection and plotted on the interactive map.

In [11]:
def _get_umm(g):
    try:
        return g.get("umm", {})
    except Exception:
        return {}


def nisar_granule_to_feature(g):
    umm = _get_umm(g)

    geom = (
        umm.get("SpatialExtent", {})
           .get("HorizontalSpatialDomain", {})
           .get("Geometry", {})
    )

    poly = None

    # Prefer polygon boundary
    gpolys = geom.get("GPolygons", [])
    if gpolys:
        pts = gpolys[0].get("Boundary", {}).get("Points", [])
        if pts:
            coords = [(p["Longitude"], p["Latitude"]) for p in pts]
            if coords and coords[0] != coords[-1]:
                coords = coords + [coords[0]]
            poly = Polygon(coords)

    # Fallback to bounding rectangle
    if poly is None:
        rects = geom.get("BoundingRectangles", [])
        if rects:
            r = rects[0]
            poly = box(
                r["WestBoundingCoordinate"],
                r["SouthBoundingCoordinate"],
                r["EastBoundingCoordinate"],
                r["NorthBoundingCoordinate"],
            )

    if poly is None:
        raise ValueError("Could not extract footprint geometry from NISAR granule metadata")

    # Time
    time_range = (
        umm.get("TemporalExtent", {})
           .get("RangeDateTime", {})
    )
    t0 = time_range.get("BeginningDateTime")
    t1 = time_range.get("EndingDateTime")

    # Title/ID-like field
    title = umm.get("GranuleUR") or umm.get("Title") or "NISAR granule"

    return {
        "type": "Feature",
        "geometry": mapping(poly),
        "properties": {"title": title, "t0": t0, "t1": t1},
    }


This cell builds a GeoJSON FeatureCollection (`nisar_fc`) from the list of NISAR search results (`nisar_results`). It loops over each returned granule `g`, converts it into a footprint polygon feature using `nisar_granule_to_feature(g)`, and appends the result to `nisar_features`. If a granule is missing usable footprint metadata (or conversion fails for any reason), it is skipped and the error is printed. Finally, all successfully created footprint features are bundled into `nisar_fc` and the cell prints how many NISAR footprint polygons are available for mapping.


In [12]:
# Build NISAR FeatureCollection for mapping
nisar_features = []
for g in nisar_results:
    try:
        nisar_features.append(nisar_granule_to_feature(g))
    except Exception as e:
        print("Skipping granule (no footprint):", e)

nisar_fc = {"type": "FeatureCollection", "features": nisar_features}
print("NISAR footprints in FeatureCollection:", len(nisar_fc["features"]))


NISAR footprints in FeatureCollection: 6


### 2) ESA BIOMASS 

This cell connects to the ESA MAAP STAC API (`STAC_URL`) using `Client.open(...)` and searches the BIOMASS STAC catalog for items that match the selected `BIOMASS_COLLECTION` and time range (`BIOMASS_DT`). It builds the search parameters in `search_kwargs`, and optionally adds a spatial filter (`bbox=BBOX`) if `USE_BIOMASS_BBOX` is set to `True`. The search is executed with `api.search(**search_kwargs)`, the returned results are collected into a Python list (`biomass_items`), and the cell prints how many BIOMASS items were fetched. If supported by the client/server, it also prints the server-side matched count (`search.matched()`), which can be larger than the number actually retrieved.


In [13]:
STAC_URL = "https://catalog.maap.eo.esa.int/catalogue/"

api = Client.open(STAC_URL)
USE_BIOMASS_BBOX = False   # set True to restrict BIOMASS to AOI, False for more items

search_kwargs = dict(
    collections=[BIOMASS_COLLECTION],
    datetime=BIOMASS_DT,
    method="GET"
)

if USE_BIOMASS_BBOX:
    search_kwargs["bbox"] = list(BBOX)

search = api.search(**search_kwargs)

biomass_items = list(search.items())
print("BIOMASS items fetched:", len(biomass_items))
try:
    print("BIOMASS items matched (server-side):", search.matched())
except Exception:
    pass


BIOMASS items fetched: 16485
BIOMASS items matched (server-side): 16485


This cell converts the list of returned BIOMASS STAC items (`biomass_items`) into a GeoJSON FeatureCollection (`biomass_fc`) that can be plotted on the Folium map. It defines a helper function `biomass_item_to_feature(item)` that extracts each item’s footprint geometry (`item.geometry`) and a few useful metadata fields from `item.properties` (item `id`, `start_datetime`, `end_datetime`, and `datetime`) into a GeoJSON Feature. It then applies this function to every item to create `biomass_features`, bundles them into `biomass_fc`, and prints how many BIOMASS footprint features are available for mapping.


In [14]:
# Build BIOMASS FeatureCollection for mapping
def biomass_item_to_feature(item):
    return {
        "type": "Feature",
        "geometry": item.geometry,
        "properties": {
            "id": item.id,
            "start_datetime": item.properties.get("start_datetime"),
            "end_datetime": item.properties.get("end_datetime"),
            "datetime": item.properties.get("datetime"),
        },
    }

biomass_features = [biomass_item_to_feature(it) for it in biomass_items]
biomass_fc = {"type": "FeatureCollection", "features": biomass_features}

print("BIOMASS footprints in FeatureCollection:", len(biomass_fc["features"]))


BIOMASS footprints in FeatureCollection: 16485


## Interactive map: NISAR and BIOMASS footprint layers

This cell defines the Area of Interest using a bounding box (`BBOX`) and uses it to center a Folium map. Here, `BBOX` is set to the full world extent (longitude −180 to 180, latitude −90 to 90), so the map is centered at (0, 0). It then creates the interactive base map (`m`) using OpenStreetMap tiles and overlays two GeoJSON layers: one for the NISAR footprint FeatureCollection (`nisar_fc`) and one for the BIOMASS footprint FeatureCollection (`biomass_fc`). Each layer is named with the number of returned items and includes tooltips showing key metadata (NISAR title/time range and BIOMASS id/time range). Finally, a `LayerControl` widget is added so you can toggle the NISAR and BIOMASS layers on and off.


In [29]:
# Create base map centered on bbox
BBOX = (-180.0, -90.0, 180.0, 90.0)

center_lat = (BBOX[1] + BBOX[3]) / 2
center_lon = (BBOX[0] + BBOX[2]) / 2

m = Map(tiles="OpenStreetMap", location=(center_lat, center_lon), zoom_start=7)

# Styles (two different colors) ---
def style_nisar(_feature):
    return {
        "color": "#1f77b4",      # outline
        "weight": 2,
        "fillColor": "#1f77b4",  # fill
        "fillOpacity": 0.15,
    }

def highlight_nisar(_feature):
    return {
        "weight": 4,
        "fillOpacity": 0.30,
    }

def style_biomass(_feature):
    return {
        "color": "#ff7f0e",
        "weight": 2,
        "fillColor": "#ff7f0e",
        "fillOpacity": 0.15,
    }

def highlight_biomass(_feature):
    return {
        "weight": 4,
        "fillOpacity": 0.30,
    }

# Add NISAR footprints (blue)
GeoJson(
    data=nisar_fc,
    name=f"NISAR ({len(nisar_fc['features'])} granules)",
    tooltip=["title", "t0", "t1"],
    style_function=style_nisar,
    highlight_function=highlight_nisar,
).add_to(m)

# Add BIOMASS footprints (orange)
GeoJson(
    data=biomass_fc,
    name=f"BIOMASS ({BIOMASS_COLLECTION}) ({len(biomass_fc['features'])} items)",
    tooltip=["id", "start_datetime", "end_datetime"],
    style_function=style_biomass,
    highlight_function=highlight_biomass,
).add_to(m)

LayerControl(collapsed=False).add_to(m)
m


## Overlap of BIOMASS tiles intersecting with NISAR granule

This cell builds an overlap (intersection) layer between the NISAR and BIOMASS STAC results using GeoPandas. It first converts each input FeatureCollection into a GeoDataFrame, optionally replacing every footprint with its bounding-box polygon (so the overlap is computed as bbox ∩ bbox, matching your earlier approach). It then runs a GeoPandas spatial join to efficiently find all NISAR–BIOMASS feature pairs that intersect. Because GeoPandas may name the “right index” column differently across versions, the code detects the correct right-side index column and uses it to pull the corresponding BIOMASS geometries. Finally, it computes the actual intersection geometry for each matched pair (producing overlap polygons), and assembles an output GeoDataFrame containing the pair indices plus key metadata fields (titles/timestamps/ids). The function returns both a GeoJSON-style FeatureCollection (overlap_fc) for mapping and a GeoDataFrame (overlap_gdf) for analysis, then prints counts of NISAR features, BIOMASS features, and total overlap pairs found.

In [23]:
def fc_to_gdf(fc, use_bbox_polygon=True, crs="EPSG:4326"):
    feats = fc.get("features", [])
    rows, geoms = [], []
    for f in feats:
        g = shape(f["geometry"])
        geom = box(*g.bounds) if use_bbox_polygon else g
        rows.append(dict(f.get("properties", {})))
        geoms.append(geom)
    return gpd.GeoDataFrame(rows, geometry=geoms, crs=crs)

def build_overlap_fc_gpd(nisar_fc, biomass_fc, use_bbox_polygon=True):
    gdf_nisar = fc_to_gdf(nisar_fc, use_bbox_polygon=use_bbox_polygon).reset_index().rename(columns={"index": "nisar_i"})
    gdf_biomass = fc_to_gdf(biomass_fc, use_bbox_polygon=use_bbox_polygon).reset_index().rename(columns={"index": "biomass_j"})

    if gdf_nisar.crs != gdf_biomass.crs:
        gdf_biomass = gdf_biomass.to_crs(gdf_nisar.crs)

    pairs = gpd.sjoin(
        gdf_nisar,
        gdf_biomass,
        how="inner",
        predicate="intersects",
        lsuffix="nisar",
        rsuffix="biomass",
    )

#Find right-index column name
    right_index_col = None
    for c in ("index_right", "index_biomass"):
        if c in pairs.columns:
            right_index_col = c
            break

    if right_index_col is None:
        candidates = [c for c in pairs.columns if c.startswith("index_")]
        if not candidates:
            raise KeyError(f"No right index column found. Columns are: {list(pairs.columns)}")
        right_index_col = candidates[0]

    # ---- Compute intersection geometry (GeoSeries vs GeoSeries) ----
    biomass_geom = gdf_biomass.loc[pairs[right_index_col], "geometry"].values
    biomass_geom = gpd.GeoSeries(biomass_geom, index=pairs.index, crs=gdf_biomass.crs)
    pairs["geometry"] = pairs.geometry.intersection(biomass_geom)

    def col(name):
        return pairs[name] if name in pairs.columns else None

    out = gpd.GeoDataFrame(
        {
            "nisar_i": pairs["nisar_i"],
            "biomass_j": pairs["biomass_j"],
            "nisar_title": col("title"),
            "nisar_t0": col("t0"),
            "nisar_t1": col("t1"),
            "biomass_id": col("id"),
            "biomass_start": col("start_datetime"),
            "biomass_end": col("end_datetime"),
            "geometry": pairs["geometry"],
        },
        geometry="geometry",
        crs=gdf_nisar.crs,
    )

    return out.__geo_interface__, out, gdf_nisar, gdf_biomass

# --- Run ---
overlap_fc, overlap_gdf, gdf_nisar, gdf_biomass = build_overlap_fc_gpd(
    nisar_fc, biomass_fc, use_bbox_polygon=True
)

print("NISAR granules:", len(nisar_fc.get("features", [])))
print("BIOMASS items:", len(biomass_fc.get("features", [])))
print("Total overlaps (pairs):", len(overlap_fc.get("features", [])))


NISAR granules: 6
BIOMASS items: 16485
Total overlaps (pairs): 73


This visualization shows only the overlap polygons (the intersection between NISAR and BIOMASS footprints). The map automatically computes the spatial extent of overlap_fc, then centers and zooms the view to that extent so the intersections are immediately visible. Tooltips are enabled on each overlap polygon to display the linked metadata fields when you hover. Since only the overlap layer is added, the original NISAR and BIOMASS layers are not shown and won’t distract from the intersection results

In [28]:
def fc_bounds(fc):
    feats = fc.get("features", [])
    if not feats:
        return None
    minx = miny = float("inf")
    maxx = maxy = float("-inf")
    for f in feats:
        g = shape(f["geometry"])
        bx, by, Bx, By = g.bounds
        minx, miny = min(minx, bx), min(miny, by)
        maxx, maxy = max(maxx, Bx), max(maxy, By)
    return (minx, miny, maxx, maxy)

def bounds_center(bounds):
    minx, miny, maxx, maxy = bounds
    return ((miny + maxy) / 2, (minx + maxx) / 2)  # (lat, lon)

# bounds for overlap only 
overlap_bounds = fc_bounds(overlap_fc)

# Fallback if overlap is empty
if overlap_bounds is None:
    overlap_bounds = (-180.0, -90.0, 180.0, 90.0)

center = bounds_center(overlap_bounds)

m = folium.Map(tiles="OpenStreetMap", location=center, zoom_start=4)

def style_overlap(_):
    return {"color": "#2ca02c", "weight": 2, "fillColor": "#2ca02c", "fillOpacity": 0.35}

def highlight(_):
    return {"weight": 4, "fillOpacity": 0.55}

GeoJson(
    data=overlap_fc,
    name=f"Overlap ({len(overlap_fc.get('features', []))} pairs)",
    tooltip=["nisar_title", "nisar_t0", "nisar_t1", "biomass_id", "biomass_start", "biomass_end"],
    style_function=style_overlap,
    highlight_function=highlight,
).add_to(m)


minx, miny, maxx, maxy = overlap_bounds
m.fit_bounds([[miny, minx], [maxy, maxx]])

m